# Text Mining Final Project - Group 60

In this project we solve three NLP tasks on the test set: Named Entity Recognition,
sentiment analysis and topic classification. For each task we try two methods from the labs
and compare them. The main comparison and error analysis is on the NER task.

- NER: spaCy and NLTK (Lab 1)
- Sentiment: VADER and Naive Bayes (Lab 3)
- Topic: a keyword method and LDA (Lab 6)

### Setup

In [1]:

# import sys
# !{sys.executable} -m pip install vaderSentiment gensim
# !{sys.executable} -m spacy download en_core_web_sm

In [2]:
import json
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

import spacy
from spacy.tokens import Doc

import nltk
from nltk import pos_tag, ne_chunk
from nltk.chunk import tree2conlltags

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# nltk data needed for tokenising, POS tagging and the NER chunker
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/hamidnatteghy/nltk_data...
[nltk_data]   Package

True

### Load the data

In [3]:

ner = pd.read_csv("NER-test.tsv", sep="\t")
sent = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")


ner.columns = [c.strip() for c in ner.columns]
sent.columns = [c.strip() for c in sent.columns]


with open("my_tweets.json") as f:
    my_tweets = json.load(f)

print("NER tokens:", len(ner))
print("test sentences:", len(sent))
print("training tweets:", len(my_tweets))
print()
print("sentiment in test set:")
print(sent["sentiment"].value_counts())
print()
print("topics in test set:")
print(sent["topic"].value_counts())

NER tokens: 214
test sentences: 10
training tweets: 50

sentiment in test set:
sentiment
positive    4
negative    3
neutral     3
Name: count, dtype: int64

topics in test set:
topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64


The test set has 10 sentences. For NER they are annotated token by token with the BIO scheme
(PER, ORG, LOC, MISC). The sentences are reviews about movies, restaurants and books. The topic
classes are not balanced (5 movie, 3 restaurant, 2 book). For Naive Bayes we use our 50 self-made
tweets from Lab 3 (17 positive, 17 negative, 16 neutral).

## Task 1: Named Entity Recognition

We compare spaCy and NLTK. We run both on the test sentences and compare the entities they find
to the gold labels. spaCy and NLTK use different label names, so first we map them to the four
labels of the test set (PER, ORG, LOC, MISC).

In [4]:
# put the gold tokens and tags of each sentence in a list
sentences = []
for sid in sorted(ner["sentence id"].unique()):
    rows = ner[ner["sentence id"] == sid]
    tokens = list(rows["token"])
    tags = list(rows["BIO NER tag"])
    sentences.append((tokens, tags))

# map the spaCy / nltk labels to the labels used in the test set
spacy_map = {"PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC",
             "NORP": "MISC", "LANGUAGE": "MISC"}
nltk_map = {"PERSON": "PER", "ORGANIZATION": "ORG", "GPE": "LOC", "LOCATION": "LOC"}

In [5]:
nlp = spacy.load("en_core_web_sm")

def spacy_tags(tokens):
    # we give spaCy the gold tokens so the tags line up with the gold tags
    doc = Doc(nlp.vocab, words=tokens)
    for name, component in nlp.pipeline:
        doc = component(doc)
    tags = []
    for token in doc:
        if token.ent_iob_ == "O" or token.ent_type_ not in spacy_map:
            tags.append("O")
        else:
            tags.append(token.ent_iob_ + "-" + spacy_map[token.ent_type_])
    return tags

def nltk_tags(tokens):
    chunks = tree2conlltags(ne_chunk(pos_tag(tokens)))
    tags = []
    for word, pos, label in chunks:
        if label == "O":
            tags.append("O")
        else:
            prefix = label[:2]          # "B-" or "I-"
            ne_type = label[2:]
            tags.append(prefix + nltk_map.get(ne_type, "MISC"))
    return tags

In [6]:
# run both taggers on every sentence and collect all the tags in one big list
gold_all, spacy_all, nltk_all = [], [], []
for tokens, tags in sentences:
    gold_all += tags
    spacy_all += spacy_tags(tokens)
    nltk_all += nltk_tags(tokens)

labels = ["B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-MISC","I-MISC"]

print("spaCy")
print(classification_report(gold_all, spacy_all, labels=labels, zero_division=0))
print("NLTK")
print(classification_report(gold_all, nltk_all, labels=labels, zero_division=0))

spaCy
              precision    recall  f1-score   support

       B-PER       0.67      0.67      0.67         6
       I-PER       1.00      0.75      0.86         8
       B-ORG       0.67      0.50      0.57         4
       I-ORG       0.75      1.00      0.86         3
       B-LOC       1.00      1.00      1.00         4
       I-LOC       1.00      1.00      1.00         2
      B-MISC       0.75      1.00      0.86         3
      I-MISC       1.00      1.00      1.00         1

   micro avg       0.83      0.81      0.82        31
   macro avg       0.85      0.86      0.85        31
weighted avg       0.84      0.81      0.82        31

NLTK
              precision    recall  f1-score   support

       B-PER       0.75      1.00      0.86         6
       I-PER       1.00      0.62      0.77         8
       B-ORG       1.00      0.50      0.67         4
       I-ORG       1.00      0.33      0.50         3
       B-LOC       0.43      0.75      0.55         4
       I-LOC 

In [7]:
# also check how many whole entities each tagger gets exactly right
def get_entities(tokens, tags):
    entities = []
    i = 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            ne_type = tags[i][2:]
            start = i
            i += 1
            while i < len(tags) and tags[i] == "I-" + ne_type:
                i += 1
            entities.append((start, i, ne_type))
        else:
            i += 1
    return entities

def entity_score(tagger):
    correct = predicted = total = 0
    for tokens, tags in sentences:
        gold = set(get_entities(tokens, tags))
        pred = set(get_entities(tokens, tagger(tokens)))
        correct += len(gold & pred)
        predicted += len(pred)
        total += len(gold)
    precision = correct / predicted
    recall = correct / total
    f1 = 2 * precision * recall / (precision + recall)
    return correct, total, round(f1, 2)

print("spaCy entities correct:", entity_score(spacy_tags))
print("NLTK entities correct:", entity_score(nltk_tags))

spaCy entities correct: (13, 17, 0.76)
NLTK entities correct: (8, 17, 0.46)


In [8]:
# print the entities side by side to see where the mistakes are
for i, (tokens, tags) in enumerate(sentences):
    gold = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, tags)]
    sp = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, spacy_tags(tokens))]
    nl = [(" ".join(tokens[s:e]), t) for s, e, t in get_entities(tokens, nltk_tags(tokens))]
    print("sentence", i)
    print("  gold :", gold)
    print("  spaCy:", sp)
    print("  nltk :", nl)

sentence 0
  gold : [('Warner Brothers', 'ORG')]
  spaCy: [('Warner Brothers', 'ORG')]
  nltk : [('Warner Brothers', 'ORG')]
sentence 1
  gold : [('New York University', 'ORG'), ('Soho', 'LOC')]
  spaCy: [('the New York University', 'ORG'), ('Soho', 'LOC')]
  nltk : [('New York University', 'LOC'), ('Soho', 'LOC')]
sentence 2
  gold : [('Italian', 'MISC')]
  spaCy: [('Italian', 'MISC')]
  nltk : [('Italian', 'LOC')]
sentence 3
  gold : [('Jane Austen', 'PER')]
  spaCy: [('Jane Austen', 'PER')]
  nltk : [('Jane Austen', 'PER')]
sentence 4
  gold : [('Carl Brashear', 'PER'), ('Cuba Gooding Jr.', 'PER'), ('African American', 'MISC'), ('Navy', 'ORG')]
  spaCy: [('Carl Brashear', 'PER'), ('Cuba Gooding Jr.', 'PER'), ('African American', 'MISC'), ('Navy', 'ORG')]
  nltk : [('Carl Brashear', 'PER'), ('Cuba Gooding', 'PER'), ('African', 'MISC'), ('American', 'LOC'), ('Navy', 'ORG')]
sentence 5
  gold : [("Chris O'Donnell", 'PER')]
  spaCy: [("Chris O'Donnell", 'PER')]
  nltk : [('Chris', 'PER'

### Results and error analysis

spaCy does clearly better than NLTK (entity F1 0.76 against 0.46).

The biggest reason is that NLTK does not have a MISC label. Because of this it gets all the
nationality and language words wrong: *Italian*, *English* and *African American* all become LOC,
which is why NLTK scores 0 on MISC. NLTK also splits some names, for example *Cuba Gooding Jr.*
becomes *Cuba Gooding* and *Chris O'Donnell* becomes only *Chris*.

spaCy's mistakes are smaller. It drops the title in *Dame Maggie Smith* and *Mr. Kruno*, it adds
*the* to *the New York University*, and it labels the Dutch restaurant *Blauwbrug* as MISC instead
of ORG. NLTK actually keeps *Mr. Kruno* together, so the two tools make different mistakes.

Both taggers are trained on news text, so they have a harder time on these review sentences and on
foreign names like *Blauwbrug*. This matches what the lectures said about models losing accuracy on
a new domain. To improve this we could train spaCy on review data, or add a list of nationalities
for NLTK so it can handle MISC.

## Task 2: Sentiment Analysis

We compare VADER (a rule based method) with Naive Bayes (a model we train ourselves). Both predict
positive, negative or neutral for each test sentence.

In [9]:
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

sent["vader"] = sent["text"].apply(vader_sentiment)

order = ["negative", "neutral", "positive"]
print(classification_report(sent["sentiment"], sent["vader"], labels=order, zero_division=0))
print(confusion_matrix(sent["sentiment"], sent["vader"], labels=order))

              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.83      0.56      0.56        10
weighted avg       0.80      0.60      0.57        10

[[1 0 2]
 [0 1 2]
 [0 0 4]]


In [10]:
# train Naive Bayes on our 50 tweets
train_texts = [t["text_of_tweet"] for t in my_tweets.values()]
train_labels = [t["sentiment_label"] for t in my_tweets.values()]

vectorizer = TfidfVectorizer(min_df=2)
X_train = vectorizer.fit_transform(train_texts)

model = MultinomialNB()
model.fit(X_train, train_labels)

# predict the test sentences
X_test = vectorizer.transform(sent["text"])
sent["nb"] = model.predict(X_test)

print(classification_report(sent["sentiment"], sent["nb"], labels=order, zero_division=0))
print(confusion_matrix(sent["sentiment"], sent["nb"], labels=order))

              precision    recall  f1-score   support

    negative       0.50      0.33      0.40         3
     neutral       0.67      0.67      0.67         3
    positive       0.40      0.50      0.44         4

    accuracy                           0.50        10
   macro avg       0.52      0.50      0.50        10
weighted avg       0.51      0.50      0.50        10

[[1 0 2]
 [0 2 1]
 [1 1 2]]


In [11]:
# compare both methods per sentence
for i in range(len(sent)):
    print("sentence", i,
          "| gold:", sent["sentiment"][i],
          "| vader:", sent["vader"][i],
          "| nb:", sent["nb"][i])

sentence 0 | gold: negative | vader: negative | nb: positive
sentence 1 | gold: positive | vader: positive | nb: positive
sentence 2 | gold: negative | vader: positive | nb: positive
sentence 3 | gold: positive | vader: positive | nb: neutral
sentence 4 | gold: neutral | vader: positive | nb: neutral
sentence 5 | gold: neutral | vader: positive | nb: positive
sentence 6 | gold: positive | vader: positive | nb: negative
sentence 7 | gold: positive | vader: positive | nb: positive
sentence 8 | gold: neutral | vader: neutral | nb: neutral
sentence 9 | gold: negative | vader: positive | nb: negative


### Results and error analysis

VADER gets 0.60 accuracy and Naive Bayes 0.50, but they make different kinds of mistakes.

VADER predicts *positive* too often: all four of its mistakes are sentences it thinks are positive
but are not (sentences 2, 4, 5 and 9). This happens with sentences like *"really trendy but they
have forgotten the food"* - VADER sees the positive word *trendy* and the *but* part is not strong
enough to change the score.

Naive Bayes makes more mixed mistakes. It was only trained on 50 short tweets, so most words in the
movie/book/restaurant sentences are new to it. For example it labels *"Blauwbrug has been our
favorite place to eat"* as negative and *"the disaster that was this movie"* as positive. So VADER
is biased towards positive, while Naive Bayes is more random because it has too little training data.

To improve this we could give VADER a rule for *but*, or train Naive Bayes on many more review
sentences instead of 50 general tweets.

## Task 3: Topic Classification

We compare a simple keyword method with LDA (topic modelling from Lab 6).

In [12]:
def keyword_topic(text):
    text = text.lower()
    if "movie" in text or "film" in text:
        return "movie"
    if "restaurant" in text or "diner" in text or "food" in text:
        return "restaurant"
    if "book" in text or "novel" in text:
        return "book"
    return "unknown"

sent["kw_topic"] = sent["text"].apply(keyword_topic)
print("keyword accuracy:", (sent["kw_topic"] == sent["topic"]).mean())
sent[["sentence id", "topic", "kw_topic"]]

keyword accuracy: 0.9


,sentence id,topic,kw_topic
0,0,movie,movie
1,1,restaurant,restaurant
2,2,restaurant,restaurant
3,3,book,book
4,4,movie,movie
5,5,movie,movie
6,6,restaurant,unknown
7,7,movie,movie
8,8,movie,movie
9,9,book,book


In [13]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

# clean each sentence into a list of useful words
texts = []
for sentence in sent["text"]:
    words = [w for w in simple_preprocess(sentence) if w not in STOPWORDS and len(w) > 2]
    texts.append(words)

dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(t) for t in texts]

lda = LdaModel(corpus, num_topics=3, id2word=dictionary, passes=30, random_state=42)
for i in range(3):
    print("topic", i, ":", [w for w, p in lda.show_topic(i)])

# give each sentence the topic with the highest probability
predicted = []
for bow in corpus:
    best = max(lda.get_document_topics(bow), key=lambda x: x[1])[0]
    predicted.append(best)

# decide which topic number is which label by looking at the words above
topic_to_label = {0: "movie", 1: "restaurant", 2: "restaurant"}
sent["lda_topic"] = [topic_to_label[t] for t in predicted]
print("LDA accuracy:", (sent["lda_topic"] == sent["topic"]).mean())

topic 0 : ['movie', 'new', 'story', 'gooding', 'wants', 'cuba', 'sea', 'focused', 'brashear', 'navy']
topic 1 : ['york', 'like', 'makes', 'young', 'university', 'fun', 'diner', 'students', 'love', 'soho']
topic 2 : ['place', 'years', 'moved', 'eat', 'blauwbrug', 'long', 'husband', 'lived', 'ago', 'favorite']
LDA accuracy: 0.7


### Results and error analysis

The keyword method gets 0.90. It only misses sentence 6 (*Blauwbrug ... favorite place to eat*)
because that sentence does not contain any of the keywords, so it returns *unknown*.

LDA is worse (0.70). With only 10 short sentences there is not enough text for LDA to find clear
topics, and because there are only 2 book sentences it never makes a separate "book" topic - the
book sentences end up mixed with the others. LDA is really made for large collections of documents,
so this result is expected. A keyword list or a trained classifier would work better on such a
small set.

## Summary

| Task | Method 1 | Method 2 | Best |
|------|----------|----------|------|
| NER | spaCy (entity F1 0.76) | NLTK (entity F1 0.46) | spaCy |
| Sentiment | VADER (acc 0.60) | Naive Bayes (acc 0.50) | VADER |
| Topic | Keyword (acc 0.90) | LDA (acc 0.70) | Keyword |

The test set is very small (10 sentences) so we cannot draw strong conclusions. The simple methods
(keyword, VADER) did surprisingly well, while the methods that need more data (Naive Bayes, LDA)
did worse because we did not have enough training data. With more time and more data we would train
the models on review text instead of general tweets.

## Division of work

- **[Hamid]**: NER code (spaCy and NLTK), NER error analysis, NER part of the poster.
- **[Zheng-An]**: sentiment code (VADER and Naive Bayes), made the tweet dataset, sentiment analysis, poster layout.
- **[Samia]**: topic code (keyword and LDA), topic analysis, data section of the poster.
- **[Behnood]**: evaluation code and tables, summary and conclusions, poster review.

All four of us worked on the code, the analysis and the poster together.